In [ ]:
import json

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
    DataCollatorForSeq2Seq
)


STUDENT_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"

DATA_FILE = "data/distilled.jsonl"

OUTPUT_DIR = "./outputs/distilled_student"


# --------------------------------------------------
# Load dataset
# --------------------------------------------------

records = []

with open(
    DATA_FILE,
    "r",
    encoding="utf-8"
) as f:

    for line in f:

        if line.strip():

            records.append(
                json.loads(line)
            )


dataset = Dataset.from_list(records)


# --------------------------------------------------
# Load tokenizer
# --------------------------------------------------

tokenizer = AutoTokenizer.from_pretrained(
    STUDENT_MODEL
)

#No need 
# --------------------------------------------------
# Convert instruction/output into training text
# --------------------------------------------------

# def format_example(example):

#     messages = [
#         {
#             "role": "user",
#             "content": example["instruction"]
#         },
#         {
#             "role": "assistant",
#             "content": example["output"]
#         }
#     ]

#     text = tokenizer.apply_chat_template(
#         messages,
#         tokenize=False,
#         add_generation_prompt=False
#     )

#     return {
#         "text": text
#     }


# dataset = dataset.map(
#     format_example
# )


# --------------------------------------------------
# Tokenization
# --------------------------------------------------

def tokenize(example):

    user_messages = [
        {
            "role": "user",
            "content": example["instruction"]
        }
    ]

    full_messages = [
        {
            "role": "user",
            "content": example["instruction"]
        },
        {
            "role": "assistant",
            "content": example["output"]
        }
    ]

    prompt_text = tokenizer.apply_chat_template(
        user_messages,
        tokenize=False,
        add_generation_prompt=True
    )

    full_text = tokenizer.apply_chat_template(
        full_messages,
        tokenize=False,
        add_generation_prompt=False
    )

    prompt_tokens = tokenizer(
        prompt_text,
        add_special_tokens=False
    )

    full_tokens = tokenizer(
        full_text,
        truncation=True,
        max_length=1024,
        add_special_tokens=False
    )

    prompt_length = len(
        prompt_tokens["input_ids"]
    )

    labels = full_tokens["input_ids"].copy()

    for i in range(
        min(prompt_length, len(labels))
    ):
        labels[i] = -100

    full_tokens["labels"] = labels

    return full_tokens


tokenized_dataset = dataset.map(
    tokenize,
    batched=True,
    remove_columns=dataset.column_names
)


# --------------------------------------------------
# Model
# --------------------------------------------------

model = AutoModelForCausalLM.from_pretrained(
    STUDENT_MODEL
)


# --------------------------------------------------
# Data collator
# --------------------------------------------------

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    padding=True,
    label_pad_token_id=-100
)

# --------------------------------------------------
# Data verification
# --------------------------------------------------


example = tokenized_dataset[0]

print("Input IDs:")
print(example["input_ids"])

print("\nLabels:")
print(example["labels"])



# --------------------------------------------------
# Training arguments
# --------------------------------------------------

training_args = TrainingArguments(

    output_dir=OUTPUT_DIR,

    num_train_epochs=3,

    per_device_train_batch_size=2,

    gradient_accumulation_steps=4,

    learning_rate=5e-5,

    logging_steps=1,

    save_strategy="epoch",

    report_to="none"
)


# --------------------------------------------------
# Trainer
# --------------------------------------------------

trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=tokenized_dataset,

    data_collator=data_collator
)


# --------------------------------------------------
# Train
# --------------------------------------------------

trainer.train()


# --------------------------------------------------
# Save
# --------------------------------------------------

trainer.save_model(
    OUTPUT_DIR
)

tokenizer.save_pretrained(
    OUTPUT_DIR
)

print(
    "Student training completed."
)